<a href="https://colab.research.google.com/github/JonasFanZ/114-2-Programing-Language/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_%E6%AD%A3%E5%BC%8F%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://docs.google.com/spreadsheets/d/1YWaNwT_PiJIH3j4PmD17BG6nhM_TUqt2KfnNCq8itTQ/edit?usp=sharing

In [11]:
!pip install -q "gradio>=4.0.0" gspread plotly pandas numpy

import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.colab import auth
from google.auth import default
import gspread
from datetime import datetime
import numpy as np

# --- 0. Google 帳號驗證與試算表連結 ---
print("正在驗證 Google 帳號...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 👇 請在這裡貼上你的 Google Sheet 完整網址
SHEET_URL = "https://docs.google.com/spreadsheets/d/1YWaNwT_PiJIH3j4PmD17BG6nhM_TUqt2KfnNCq8itTQ/edit?usp=sharing"

try:
    sh = gc.open_by_url(SHEET_URL)
    print("✅ 成功連結到指定的 Google Sheet！")
    ws_wallets = sh.worksheet("Wallets")
    ws_trans = sh.worksheet("Transactions")

    # 動態檢查與建立 Categories 工作表
    try:
        ws_categories = sh.worksheet("Categories")
    except Exception:
        print("建立 Categories 工作表...")
        ws_categories = sh.add_worksheet(title="Categories", rows="100", cols="2")
        ws_categories.append_row(["Type", "Category_Name"])
        for row in [["支出", "飲食"], ["支出", "交通"], ["支出", "娛樂"], ["支出", "購物"], ["支出", "其他"],
                    ["收入", "薪資"], ["收入", "投資"], ["收入", "獎金"], ["收入", "其他"]]:
            ws_categories.append_row(row)

except Exception as e:
    print(f"❌ 連結失敗，請確認網址正確且工作表名稱設定無誤。錯誤: {e}")

# --- 1. 核心資料讀取與處理 ---
def get_wallets_data():
    try:
        records = ws_wallets.get_all_records()
        df = pd.DataFrame(records) if records else pd.DataFrame(columns=["Wallet_Name", "Balance"])
        if not df.empty:
            df['Balance'] = df['Balance'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
            df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce').fillna(0)
        return df
    except Exception: return pd.DataFrame(columns=["Wallet_Name", "Balance"])

def get_trans_data():
    try:
        records = ws_trans.get_all_records()
        df = pd.DataFrame(records) if records else pd.DataFrame(columns=["Date", "Type", "Amount", "Category", "From_Wallet", "To_Wallet", "Note"])
        if not df.empty:
            df['Amount'] = df['Amount'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
            df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        return df
    except Exception: return pd.DataFrame(columns=["Date", "Type", "Amount", "Category", "From_Wallet", "To_Wallet", "Note"])

def get_wallet_choices():
    df = get_wallets_data()
    return df['Wallet_Name'].tolist() if not df.empty else []

def get_categories(cat_type):
    try:
        records = ws_categories.get_all_records()
        df = pd.DataFrame(records)
        if not df.empty:
            return df[df['Type'] == cat_type]['Category_Name'].tolist()
    except: pass
    return ["飲食", "交通", "娛樂", "購物", "其他"] if cat_type == '支出' else ["薪資", "投資", "獎金", "其他"]

def get_trans_data_for_ui(wallet_filter="全部"):
    df = get_trans_data()
    if df.empty:
        return pd.DataFrame(columns=["Date", "Type", "Amount", "Category", "From_Wallet", "To_Wallet", "Note"])
    if wallet_filter != "全部":
        df = df[(df['From_Wallet'] == wallet_filter) | (df['To_Wallet'] == wallet_filter)]
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    df = df.sort_values(by="Date", ascending=False)
    return df

# --- 2. 數據看板計算 ---
def get_dashboard_metrics():
    df_trans = get_trans_data()
    df_wallets = get_wallets_data()

    current_nw = df_wallets['Balance'].sum() if not df_wallets.empty else 0
    today = pd.Timestamp.today()
    this_month_start = today.replace(day=1)
    last_month_start = (this_month_start - pd.DateOffset(days=1)).replace(day=1)
    last_month_end = this_month_start - pd.DateOffset(days=1)

    if not df_trans.empty:
        expenses = df_trans[df_trans['Type'] == '支出']
        exp_this_month = expenses[expenses['Date'] >= this_month_start]['Amount'].sum()
        exp_last_month = expenses[(expenses['Date'] >= last_month_start) & (expenses['Date'] <= last_month_end)]['Amount'].sum()

        this_month_trans = df_trans[df_trans['Date'] >= this_month_start]
        net_change_this_month = this_month_trans[this_month_trans['Type'] == '收入']['Amount'].sum() - this_month_trans[this_month_trans['Type'] == '支出']['Amount'].sum()
        last_month_nw = current_nw - net_change_this_month
    else:
        exp_this_month = exp_last_month = 0
        last_month_nw = current_nw

    def format_diff(current, previous, invert_color=False):
        if previous == 0: return "<span class='diff neutral'>--</span>"
        pct = ((current - previous) / previous) * 100
        sign = "+" if pct > 0 else ""
        if pct > 0: color_class = 'negative' if invert_color else 'positive'
        elif pct < 0: color_class = 'positive' if invert_color else 'negative'
        else: color_class = 'neutral'
        return f"<span class='diff {color_class}'>{sign}{pct:.1f}% (與上月)</span>"

    nw_html = f"<div class='metric-title'>總淨資產</div><div class='metric-value'>NT$ {current_nw:,.0f}</div>{format_diff(current_nw, last_month_nw)}"
    exp_html = f"<div class='metric-title'>本月總支出</div><div class='metric-value'>NT$ {exp_this_month:,.0f}</div>{format_diff(exp_this_month, exp_last_month, invert_color=True)}"
    return nw_html, exp_html

# --- 3. 圖表生成邏輯 (三圖版 - 中心加總強化版) ---
def update_charts():
    try:
        df_wallets = get_wallets_data()
        df_trans = get_trans_data()
        total_now = float(df_wallets['Balance'].sum()) if not df_wallets.empty else 0.0

        # 1. 資產分布圓餅圖
        df_pie = df_wallets[df_wallets['Balance'] > 0] if not df_wallets.empty else pd.DataFrame()
        if not df_pie.empty:
            # 計算加總數字 (Req)
            total_assets_str = f"NT$ {df_wallets['Balance'].sum():,.0f}"

            fig_pie = px.pie(df_pie, values='Balance', names='Wallet_Name', hole=0.6, color_discrete_sequence=px.colors.qualitative.Pastel)
            fig_pie.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font_color='#1e293b', margin=dict(t=20, b=20, l=20, r=20), showlegend=False)
            fig_pie.update_traces(textposition='inside', textinfo='percent+label')

            # 🌟 新增：中心加總數字註解
            fig_pie.add_annotation(
                text=f"總資產<br><b style='font-size:20px; color:#1e293b;'>{total_assets_str}</b>",
                x=0.5, y=0.5, showarrow=False, font=dict(family="Optima", color="#64748b", size=14)
            )
        else:
            fig_pie = go.Figure().update_layout(title="尚無正數資產", paper_bgcolor='rgba(0,0,0,0)', font_color='#1e293b')

        # 2. 本月支出圓餅圖
        today = pd.Timestamp.today()
        this_month_start = today.replace(day=1)
        if not df_trans.empty:
            this_month_exp = df_trans[(df_trans['Type'] == '支出') & (df_trans['Date'] >= this_month_start)]
            if not this_month_exp.empty and this_month_exp['Amount'].sum() > 0:

                # 計算加總數字 (Req)
                total_expense_str = f"NT$ {this_month_exp['Amount'].sum():,.0f}"

                fig_exp = px.pie(this_month_exp, values='Amount', names='Category', hole=0.5, color_discrete_sequence=px.colors.qualitative.Set2)
                fig_exp.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font_color='#1e293b', margin=dict(t=20, b=20, l=20, r=20), showlegend=False)
                fig_exp.update_traces(textposition='inside', textinfo='percent+label')

                # 🌟 新增：中心加總數字註解
                fig_exp.add_annotation(
                    text=f"本月支出<br><b style='font-size:18px; color:#1e293b;'>{total_expense_str}</b>",
                    x=0.5, y=0.5, showarrow=False, font=dict(family="Optima", color="#64748b", size=14)
                )
            else:
                fig_exp = go.Figure().update_layout(title="本月尚無支出紀錄", paper_bgcolor='rgba(0,0,0,0)', font_color='#1e293b')
        else:
            fig_exp = go.Figure().update_layout(title="本月尚無支出紀錄", paper_bgcolor='rgba(0,0,0,0)', font_color='#1e293b')

        # 3. 走勢圖 (維持原樣)
        if not df_trans.empty:
            impact_df = df_trans.copy()
            impact_df['Impact'] = np.where(impact_df['Type'] == '收入', impact_df['Amount'], np.where(impact_df['Type'] == '支出', -impact_df['Amount'], 0))
            daily_impact = impact_df.groupby('Date')['Impact'].sum()
            min_date = daily_impact.index.min()
            today_ts = pd.Timestamp.today().normalize()
            if pd.isna(min_date): min_date = today_ts
            all_dates = pd.date_range(start=min_date, end=today_ts)
            daily_impact = daily_impact.reindex(all_dates, fill_value=0)
            cumulative_impact = daily_impact.cumsum()
            total_impact = cumulative_impact.iloc[-1] if not cumulative_impact.empty else 0
            initial_nw = total_now - total_impact
            nw_series = initial_nw + cumulative_impact

            fig_line = px.line(x=nw_series.index.strftime('%Y-%m-%d').tolist(), y=nw_series.values.tolist(), markers=True)
        else:
            fig_line = px.line(x=[pd.Timestamp.today().strftime('%Y-%m-%d')], y=[total_now], markers=True)

        fig_line.update_traces(line_color='#3b82f6', line_width=3)
        fig_line.update_layout(
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font_color='#1e293b',
            margin=dict(t=20, b=20, l=20, r=20), xaxis_title="", yaxis_title="淨資產",
            xaxis=dict(
                rangeselector=dict(buttons=list([
                    dict(count=7, label="1週", step="day", stepmode="backward"),
                    dict(count=1, label="1個月", step="month", stepmode="backward"),
                    dict(count=3, label="3個月", step="month", stepmode="backward"),
                    dict(step="all", label="全部")
                ])),
                rangeslider=dict(visible=False), type="date"
            )
        )
        return fig_line, fig_pie, fig_exp
    except Exception as e:
        print(f"圖表錯誤: {e}")
        err_fig = go.Figure().update_layout(title="圖表渲染失敗", paper_bgcolor='rgba(0,0,0,0)')
        return err_fig, err_fig, err_fig

# --- 4. 寫入與編輯邏輯 ---
def parse_date(date_val):
    if not date_val: return datetime.today().strftime('%Y-%m-%d')
    try:
        if isinstance(date_val, str) and date_val.replace('.', '', 1).isdigit(): date_val = float(date_val)
        if isinstance(date_val, (int, float)):
            if date_val > 1e11: return pd.to_datetime(date_val, unit='ms').strftime('%Y-%m-%d')
            return pd.to_datetime(date_val, unit='s').strftime('%Y-%m-%d')
        return pd.to_datetime(date_val).strftime('%Y-%m-%d')
    except Exception: return datetime.today().strftime('%Y-%m-%d')

def add_record(record_type, date_val, amount, category, from_w, to_w, note, current_filter):
    amount = int(amount)
    date_str = parse_date(date_val)
    ws_trans.append_row([date_str, record_type, amount, category, from_w, to_w, note])

    df_wallets = get_wallets_data()
    if record_type == "支出":
        idx = df_wallets.index[df_wallets['Wallet_Name'] == from_w].tolist()[0]
        ws_wallets.update_cell(idx + 2, 2, int(df_wallets.at[idx, 'Balance']) - amount)
    elif record_type == "收入":
        idx = df_wallets.index[df_wallets['Wallet_Name'] == to_w].tolist()[0]
        ws_wallets.update_cell(idx + 2, 2, int(df_wallets.at[idx, 'Balance']) + amount)
    elif record_type == "轉移":
        idx_from = df_wallets.index[df_wallets['Wallet_Name'] == from_w].tolist()[0]
        idx_to = df_wallets.index[df_wallets['Wallet_Name'] == to_w].tolist()[0]
        ws_wallets.update_cell(idx_from + 2, 2, int(df_wallets.at[idx_from, 'Balance']) - amount)
        ws_wallets.update_cell(idx_to + 2, 2, int(df_wallets.at[idx_to, 'Balance']) + amount)

    return gr.update(visible=False), "✅ 記錄成功！", *get_dashboard_metrics(), *update_charts(), get_trans_data_for_ui(current_filter)

def add_new_wallet(wallet_name, initial_balance, current_filter):
    if not wallet_name: return gr.update(), "❌ 請輸入名稱", *get_dashboard_metrics(), *update_charts(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), get_trans_data_for_ui(current_filter)
    df_wallets = get_wallets_data()
    if wallet_name in df_wallets['Wallet_Name'].values: return gr.update(), f"❌ 錢包已存在！", *get_dashboard_metrics(), *update_charts(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), get_trans_data_for_ui(current_filter)

    ws_wallets.append_row([wallet_name, int(initial_balance)])
    ws_trans.append_row([datetime.today().strftime('%Y-%m-%d'), "收入", int(initial_balance), "初始餘額", "", wallet_name, "建立錢包"])
    new_choices = get_wallet_choices()
    dd_update = gr.update(choices=new_choices)
    ws_update = gr.update(choices=["全部"] + new_choices)
    return gr.update(visible=False), f"✅ 新增錢包：{wallet_name}", *get_dashboard_metrics(), *update_charts(), dd_update, dd_update, dd_update, dd_update, ws_update, get_trans_data_for_ui(current_filter)

def add_new_category(cat_type, cat_name):
    if not cat_name: return gr.update(), "❌ 請輸入分類名稱", gr.update(), gr.update()
    existing = get_categories(cat_type)
    if cat_name in existing: return gr.update(), f"❌ 分類「{cat_name}」已存在！", gr.update(), gr.update()

    ws_categories.append_row([cat_type, cat_name])
    return gr.update(visible=False), f"✅ 新增{cat_type}分類：{cat_name}", gr.update(choices=get_categories('支出')), gr.update(choices=get_categories('收入'))

def filter_table(wallet):
    return get_trans_data_for_ui(wallet), gr.update(visible=(wallet == "全部")), gr.update(value="⚠️ *僅在選擇「全部」時開放編輯與刪除。*" if wallet != "全部" else "")

def get_wallet_impacts(df):
    impacts = {}
    if df.empty: return impacts
    df = df.copy()
    df['Amount'] = df['Amount'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
    df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)
    for _, row in df.iterrows():
        amt = row['Amount']
        t_type = str(row['Type']).strip()
        f_w, t_w = str(row.get('From_Wallet', '')).strip(), str(row.get('To_Wallet', '')).strip()
        if t_type == '收入' and t_w: impacts[t_w] = impacts.get(t_w, 0) + amt
        elif t_type == '支出' and f_w: impacts[f_w] = impacts.get(f_w, 0) - amt
        elif t_type == '轉移':
            if f_w: impacts[f_w] = impacts.get(f_w, 0) - amt
            if t_w: impacts[t_w] = impacts.get(t_w, 0) + amt
    return impacts

def save_edited_transactions(edited_df):
    try:
        old_impacts = get_wallet_impacts(get_trans_data())
        new_impacts = get_wallet_impacts(edited_df)
        df_w = get_wallets_data()
        for i, row in df_w.iterrows():
            w_name = str(row['Wallet_Name']).strip()
            diff = new_impacts.get(w_name, 0) - old_impacts.get(w_name, 0)
            if diff != 0: ws_wallets.update_cell(i + 2, 2, int(float(row['Balance']) + diff))

        ws_trans.clear()
        edited_df['Date'] = pd.to_datetime(edited_df['Date']).dt.strftime('%Y-%m-%d')
        ws_trans.update([edited_df.columns.values.tolist()] + edited_df.values.tolist())
        return "✅ 明細修改並校正完畢！", *get_dashboard_metrics(), *update_charts()
    except Exception as e: return f"❌ 儲存失敗: {e}", *get_dashboard_metrics(), *update_charts()

# --- 5. Gradio UI 介面設計 ---
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&display=swap');
body, .gradio-container { background-color: #f8fafc; color: #1e293b; font-family: 'Optima', sans-serif; }
.glass-panel { background-color: #ffffff !important; border-radius: 12px; border: 1px solid #e2e8f0; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05); padding: 20px; margin-bottom: 15px; }
.metric-box { background-color: #ffffff; border-radius: 12px; border: 1px solid #e2e8f0; padding: 20px; text-align: center; box-shadow: 0 2px 4px -1px rgba(0,0,0,0.03); }
.metric-title { font-size: 1.1rem; color: #64748b; font-weight: bold; margin-bottom: 5px; }
.metric-value { font-size: 2.5rem; font-weight: bold; color: #1e293b; font-family: 'IBM Plex Mono', monospace; letter-spacing: -1px; margin-bottom: 8px;}
.diff { font-size: 0.9rem; font-weight: bold; padding: 4px 8px; border-radius: 20px; }
.positive { background-color: #dcfce7; color: #166534; }
.negative { background-color: #fee2e2; color: #991b1b; }
.neutral { background-color: #f1f5f9; color: #475569; }
button.primary { background-color: #3b82f6 !important; color: #ffffff !important; font-weight: bold; border-radius: 8px; border: none; }
button.primary:hover { background-color: #2563eb !important; }
button.fab { background-color: #1e293b !important; color: white !important; font-size: 1.2rem; border-radius: 50px; padding: 10px 20px; margin: 10px auto; display: block; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1); }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as demo:


    with gr.Row():
        nw_html, exp_html = get_dashboard_metrics()
        nw_display = gr.HTML(f"<div class='metric-box'>{nw_html}</div>")
        exp_display = gr.HTML(f"<div class='metric-box'>{exp_html}</div>")

    status_msg = gr.Markdown("")
    btn_toggle_action = gr.Button("➕ 記帳 / 新增項目", elem_classes=["fab"])

    with gr.Column(visible=False, elem_classes=["glass-panel"]) as action_panel:
        with gr.Tabs():
            with gr.TabItem("📉 支出"):
                exp_date = gr.DateTime(label="📅 點擊選擇日期", include_time=False)
                exp_wallet = gr.Dropdown(choices=get_wallet_choices(), label="扣款錢包")
                exp_cat = gr.Dropdown(choices=get_categories('支出'), label="分類")
                exp_amt = gr.Number(label="金額", value=0)
                exp_note = gr.Textbox(label="備註")
                btn_exp = gr.Button("確認支出", variant="primary")
            with gr.TabItem("📈 收入"):
                inc_date = gr.DateTime(label="📅 點擊選擇日期", include_time=False)
                inc_wallet = gr.Dropdown(choices=get_wallet_choices(), label="入帳錢包")
                inc_cat = gr.Dropdown(choices=get_categories('收入'), label="分類")
                inc_amt = gr.Number(label="金額", value=0)
                inc_note = gr.Textbox(label="備註")
                btn_inc = gr.Button("確認收入", variant="primary")
            with gr.TabItem("🔄 轉移"):
                trf_date = gr.DateTime(label="📅 點擊選擇日期", include_time=False)
                trf_from = gr.Dropdown(choices=get_wallet_choices(), label="轉出錢包")
                trf_to = gr.Dropdown(choices=get_wallet_choices(), label="轉入錢包")
                trf_amt = gr.Number(label="金額", value=0)
                trf_note = gr.Textbox(label="備註")
                btn_trf = gr.Button("確認轉移", variant="primary")
            with gr.TabItem("💼 新增錢包"):
                new_w_name = gr.Textbox(label="新錢包名稱")
                new_w_bal = gr.Number(label="初始餘額", value=0)
                btn_add_w = gr.Button("建立錢包", variant="primary")
            with gr.TabItem("🏷️ 新增分類"):
                new_c_type = gr.Radio(choices=["支出", "收入"], label="分類類型", value="支出")
                new_c_name = gr.Textbox(label="新分類名稱", placeholder="例如：遊戲課金")
                btn_add_c = gr.Button("建立分類", variant="primary")
        btn_close_action = gr.Button("收起面板 🔼")

    # 雙欄式版面配置 (2:1)
    with gr.Row():
        # 左欄：時間軸與明細 (Scale=2)
        with gr.Column(scale=2):
            with gr.Column(elem_classes=["glass-panel"]):
                gr.Markdown("<h4 style='color: #1e293b; margin:0;'>📈 淨資產走勢</h4>")
                line_chart = gr.Plot()
            with gr.Column(elem_classes=["glass-panel"]):
                gr.Markdown("<h4 style='color: #1e293b; margin:0;'>📝 歷史明細</h4>")
                wallet_selector = gr.Radio(choices=["全部"] + get_wallet_choices(), value="全部", label="過濾")
                edit_warning = gr.Markdown("")
                full_trans_table = gr.Dataframe(value=get_trans_data_for_ui(), interactive=True, type="pandas")
                btn_save_edits = gr.Button("💾 儲存修改", variant="primary")

        # 右欄：組成比例分析 (Scale=1)
        with gr.Column(scale=1):
            with gr.Column(elem_classes=["glass-panel"]):
                gr.Markdown("<h4 style='color: #1e293b; margin:0;'>🍩 資產分布</h4>")
                pie_chart = gr.Plot()
            with gr.Column(elem_classes=["glass-panel"]):
                gr.Markdown("<h4 style='color: #1e293b; margin:0;'>📊 本月支出分類</h4>")
                exp_pie_chart = gr.Plot()

    btn_toggle_action.click(fn=lambda: gr.update(visible=True), outputs=[action_panel])
    btn_close_action.click(fn=lambda: gr.update(visible=False), outputs=[action_panel])

    demo.load(update_charts, inputs=[], outputs=[line_chart, pie_chart, exp_pie_chart])

    wallet_selector.change(fn=filter_table, inputs=[wallet_selector], outputs=[full_trans_table, btn_save_edits, edit_warning])

    btn_exp.click(fn=lambda d, a, c, w, n, f: add_record("支出", d, a, c, w, "", n, f), inputs=[exp_date, exp_amt, exp_cat, exp_wallet, exp_note, wallet_selector], outputs=[action_panel, status_msg, nw_display, exp_display, line_chart, pie_chart, exp_pie_chart, full_trans_table])
    btn_inc.click(fn=lambda d, a, c, w, n, f: add_record("收入", d, a, c, "", w, n, f), inputs=[inc_date, inc_amt, inc_cat, inc_wallet, inc_note, wallet_selector], outputs=[action_panel, status_msg, nw_display, exp_display, line_chart, pie_chart, exp_pie_chart, full_trans_table])
    btn_trf.click(fn=lambda d, a, wf, wt, n, f: add_record("轉移", d, a, "資產轉移", wf, wt, n, f), inputs=[trf_date, trf_amt, trf_from, trf_to, trf_note, wallet_selector], outputs=[action_panel, status_msg, nw_display, exp_display, line_chart, pie_chart, exp_pie_chart, full_trans_table])
    btn_add_w.click(fn=add_new_wallet, inputs=[new_w_name, new_w_bal, wallet_selector], outputs=[action_panel, status_msg, nw_display, exp_display, line_chart, pie_chart, exp_pie_chart, exp_wallet, inc_wallet, trf_from, trf_to, wallet_selector, full_trans_table])

    btn_add_c.click(fn=add_new_category, inputs=[new_c_type, new_c_name], outputs=[action_panel, status_msg, exp_cat, inc_cat])

    btn_save_edits.click(fn=save_edited_transactions, inputs=[full_trans_table], outputs=[status_msg, nw_display, exp_display, line_chart, pie_chart, exp_pie_chart])

demo.launch(debug=True, share=True)

正在驗證 Google 帳號...
✅ 成功連結到指定的 Google Sheet！


/tmp/ipykernel_811/3829172314.py:315: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_811/3829172314.py:315: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3261af11ea1135a20b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3261af11ea1135a20b.gradio.live
